# UK Train Rides Dataset - Data Cleaning

This notebook focuses on handling missing values, validating data quality, and preparing the dataset for preprocessing and analysis.

In [1]:
import pandas as pd

df = pd.read_csv("railway.csv")

df.head()

,Transaction ID,Date of Purchase,Time of Purchase,Purchase Type,Payment Method,Railcard,Ticket Class,Ticket Type,Price,Departure Station,Arrival Destination,Date of Journey,Departure Time,Arrival Time,Actual Arrival Time,Journey Status,Reason for Delay,Refund Request
0,da8a6ba8-b3dc-4677-b176,2023-12-08,12:41:11,Online,Contactless,Adult,Standard,Advance,43,London Paddington,Liverpool Lime Street,2024-01-01,11:00:00,13:30:00,13:30:00,On Time,NaN,No
1,b0cdd1b0-f214-4197-be53,2023-12-16,11:23:01,Station,Credit Card,Adult,Standard,Advance,23,London Kings Cross,York,2024-01-01,09:45:00,11:35:00,11:40:00,Delayed,Signal Failure,No
2,f3ba7a96-f713-40d9-9629,2023-12-19,19:51:27,Online,Credit Card,NaN,Standard,Advance,3,Liverpool Lime Street,Manchester Piccadilly,2024-01-02,18:15:00,18:45:00,18:45:00,On Time,NaN,No
3,b2471f11-4fe7-4c87-8ab4,2023-12-20,23:00:36,Station,Credit Card,NaN,Standard,Advance,13,London Paddington,Reading,2024-01-01,21:30:00,22:30:00,22:30:00,On Time,NaN,No
4,2be00b45-0762-485e-a7a3,2023-12-27,18:22:56,Online,Contactless,NaN,Standard,Advance,76,Liverpool Lime Street,London Euston,2024-01-01,16:45:00,19:00:00,19:00:00,On Time,NaN,No


## Missing Values Analysis

In [2]:
df.isnull().sum()

Transaction ID             0
Date of Purchase           0
Time of Purchase           0
Purchase Type              0
Payment Method             0
Railcard               20918
Ticket Class               0
Ticket Type                0
Price                      0
Departure Station          0
Arrival Destination        0
Date of Journey            0
Departure Time             0
Arrival Time               0
Actual Arrival Time     1880
Journey Status             0
Reason for Delay       27481
Refund Request             0
dtype: int64

In [3]:
(df.isnull().sum() / len(df)) * 100

Transaction ID          0.000000
Date of Purchase        0.000000
Time of Purchase        0.000000
Purchase Type           0.000000
Payment Method          0.000000
Railcard               66.085363
Ticket Class            0.000000
Ticket Type             0.000000
Price                   0.000000
Departure Station       0.000000
Arrival Destination     0.000000
Date of Journey         0.000000
Departure Time          0.000000
Arrival Time            0.000000
Actual Arrival Time     5.939405
Journey Status          0.000000
Reason for Delay       86.819575
Refund Request          0.000000
dtype: float64

In [4]:
df.shape

(31653, 18)

## Creating a Working Copy

In [5]:
df_clean = df.copy()

## Railcard Analysis

In [6]:
df_clean["Railcard"].value_counts(dropna=False)

Railcard
NaN         20918
Adult        4846
Disabled     3089
Senior       2800
Name: count, dtype: int64

## Actual Arrival Time Analysis

In [7]:
df_clean["Actual Arrival Time"].isnull().sum()

np.int64(1880)

In [8]:
df_clean[df_clean["Actual Arrival Time"].isnull()]["Journey Status"].value_counts()

Journey Status
Cancelled    1880
Name: count, dtype: int64

## Reason for Delay Analysis

In [9]:
df_clean[df_clean["Reason for Delay"].isnull()]["Journey Status"].value_counts()

Journey Status
On Time    27481
Name: count, dtype: int64

In [10]:
df_clean["Railcard"] = df_clean["Railcard"].fillna("No Railcard")

In [11]:
df_clean["Railcard"].value_counts(dropna=False)

Railcard
No Railcard    20918
Adult           4846
Disabled        3089
Senior          2800
Name: count, dtype: int64

In [12]:
df_clean.isnull().sum()

Transaction ID             0
Date of Purchase           0
Time of Purchase           0
Purchase Type              0
Payment Method             0
Railcard                   0
Ticket Class               0
Ticket Type                0
Price                      0
Departure Station          0
Arrival Destination        0
Date of Journey            0
Departure Time             0
Arrival Time               0
Actual Arrival Time     1880
Journey Status             0
Reason for Delay       27481
Refund Request             0
dtype: int64

## Cleaning Decisions

- Missing values in Railcard were replaced with "No Railcard".
- Missing values in Actual Arrival Time were retained because they correspond to cancelled journeys.
- Missing values in Reason for Delay were retained because on-time journeys do not have delay reasons.
- No duplicate records were found.

## Date and Time Conversion

In [13]:
df_clean.dtypes

Transaction ID           str
Date of Purchase         str
Time of Purchase         str
Purchase Type            str
Payment Method           str
Railcard                 str
Ticket Class             str
Ticket Type              str
Price                  int64
Departure Station        str
Arrival Destination      str
Date of Journey          str
Departure Time           str
Arrival Time             str
Actual Arrival Time      str
Journey Status           str
Reason for Delay         str
Refund Request           str
dtype: object

In [14]:
df_clean["Date of Purchase"] = pd.to_datetime(df_clean["Date of Purchase"])

df_clean["Date of Journey"] = pd.to_datetime(df_clean["Date of Journey"])

In [15]:
df_clean[["Date of Purchase","Date of Journey"]].dtypes

Date of Purchase    datetime64[us]
Date of Journey     datetime64[us]
dtype: object

In [16]:
df_clean["Time of Purchase"] = pd.to_datetime(
    df_clean["Time of Purchase"],
    format="%H:%M:%S"
)

df_clean["Departure Time"] = pd.to_datetime(
    df_clean["Departure Time"],
    format="%H:%M:%S"
)

df_clean["Arrival Time"] = pd.to_datetime(
    df_clean["Arrival Time"],
    format="%H:%M:%S"
)

df_clean["Actual Arrival Time"] = pd.to_datetime(
    df_clean["Actual Arrival Time"],
    format="%H:%M:%S",
    errors="coerce"
)

In [17]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Transaction ID       31653 non-null  str           
 1   Date of Purchase     31653 non-null  datetime64[us]
 2   Time of Purchase     31653 non-null  datetime64[us]
 3   Purchase Type        31653 non-null  str           
 4   Payment Method       31653 non-null  str           
 5   Railcard             31653 non-null  str           
 6   Ticket Class         31653 non-null  str           
 7   Ticket Type          31653 non-null  str           
 8   Price                31653 non-null  int64         
 9   Departure Station    31653 non-null  str           
 10  Arrival Destination  31653 non-null  str           
 11  Date of Journey      31653 non-null  datetime64[us]
 12  Departure Time       31653 non-null  datetime64[us]
 13  Arrival Time         31653 non-null  datet

## Price Analysis and Outlier Detection

In [18]:
df_clean["Price"].describe()

count    31653.000000
mean        23.439200
std         29.997628
min          1.000000
25%          5.000000
50%         11.000000
75%         35.000000
max        267.000000
Name: Price, dtype: float64

In [19]:
Q1 = df_clean["Price"].quantile(0.25)
Q3 = df_clean["Price"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

Lower Bound: -40.0
Upper Bound: 80.0


In [20]:
outliers = df_clean[
    (df_clean["Price"] < lower_bound) |
    (df_clean["Price"] > upper_bound)
]

outliers.shape

(1555, 18)

In [21]:
outliers.head()

,Transaction ID,Date of Purchase,Time of Purchase,Purchase Type,Payment Method,Railcard,Ticket Class,Ticket Type,Price,Departure Station,Arrival Destination,Date of Journey,Departure Time,Arrival Time,Actual Arrival Time,Journey Status,Reason for Delay,Refund Request
25,842da93c-b820-42dc-ad4f,2023-12-31,1900-01-01 15:19:53,Online,Contactless,No Railcard,Standard,Advance,86,Manchester Piccadilly,London Paddington,2024-01-01,1900-01-01 13:45:00,1900-01-01 16:00:00,1900-01-01 16:00:00,On Time,NaN,No
45,767314a0-f839-4607-a3d3,2024-01-01,1900-01-01 05:09:30,Station,Credit Card,No Railcard,First Class,Advance,134,Manchester Piccadilly,London Euston,2024-01-02,1900-01-01 03:30:00,1900-01-01 05:20:00,1900-01-01 05:31:00,Delayed,Weather Conditions,No
51,382d60f9-9fe0-4920-97e4,2024-01-01,1900-01-01 06:34:08,Station,Credit Card,No Railcard,Standard,Anytime,151,Liverpool Lime Street,London Euston,2024-01-01,1900-01-01 08:00:00,1900-01-01 10:15:00,1900-01-01 10:39:00,Delayed,Weather,No
61,711c08ba-eb61-44ba-821a,2024-01-01,1900-01-01 09:30:09,Station,Credit Card,No Railcard,First Class,Advance,134,Manchester Piccadilly,London Euston,2024-01-02,1900-01-01 08:00:00,1900-01-01 09:50:00,1900-01-01 10:08:00,Delayed,Weather,No
68,9082a416-480e-4ca4-bf9d,2024-01-01,1900-01-01 15:39:11,Station,Credit Card,No Railcard,Standard,Anytime,151,Liverpool Lime Street,London Euston,2024-01-01,1900-01-01 17:00:00,1900-01-01 19:15:00,1900-01-01 19:15:00,On Time,NaN,No


## Outlier Decision

Outliers were detected in the Price column using the IQR method.

These values were retained because they represent valid high-priced train tickets rather than data entry errors.

Removing them could result in the loss of important business information.

In [22]:
df_clean.to_csv(
    "railway_cleaned.csv",
    index=False
)

In [23]:
print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


## Data Cleaning Summary

- Missing values in Railcard were replaced with "No Railcard".
- Date columns were converted to datetime format.
- Time columns were converted to datetime format.
- Missing values in Actual Arrival Time were retained because they correspond to cancelled journeys.
- Missing values in Reason for Delay were retained because they correspond to on-time journeys.
- No duplicate records were found.
- Outliers were identified in the Price column for further evaluation.